# Observabilidad

## Ejemplo

Tenemos el sistema de tanques del ejemplo de la sesión de control por realimentación del vector de estado.
![alt text](image.png)


In [ ]:
using ControlSystems, LinearAlgebra, Plots, LaTeXStrings

In [ ]:
A = [-1.0  1.0;
     0.5  -1.0];
B = [1.0; 0.0;;];   # columna (2×1)
C = [0.0 1.0];      # medimos h₂
D = 0.0;

sistema = ss(A, B, C, D)

### Test de observabilidad

Según lo visto, un sistema es *observable* si y solo si la matriz

$$\mathcal{W}_o=\begin{bmatrix} C \\ CA \\ \vdots \\ CA^{n-1}\end{bmatrix}$$

tiene rango $n$.

Para nuestro sistema de los tanques:

In [ ]:
Wo = obsv(A, C)
println("Wo = "); display(Wo)
println("det(Wo) = ", det(Wo))
println("rank(Wo) = ", rank(Wo))

Como el determinante de la matriz de observabilidad es $\det(\mathcal{O})=-0.5\neq 0$, el sistema es observable. Miremos los polos del sistema:

In [ ]:
polos = eigvals(A)

Vamos a diseñar un observador para el sistema con polos en $s_{1,2}$ diez veces más rápidos que los de lazo abierto:

In [ ]:
polos_obs = polos * 20
L = Matrix(place(A', C', polos_obs)')   # ganancia del observador (dualidad)

## Simulación del funcionamiento del observador

Como hemos visto, un observador es definido por la siguiente ecuación de estado:

$$\dot{\hat{x}}= (A-LC)\,\hat{x}+\begin{bmatrix}B & L\end{bmatrix}\begin{bmatrix}u \\ y\end{bmatrix}$$

A continuación definimos el observador como sistema lineal (entradas: $u$ y $y$; salidas: los estados estimados $\hat{x}$):

In [ ]:
observador = ss(A - L*C, [B L], I(2), zeros(2, 2))

Construimos el **sistema aumentado** (planta + observador) para simular el funcionamiento del observador. Definiendo el estado aumentado $z = [x;\ \hat{x}]$:

$$\dot{z} = \begin{bmatrix} A & 0 \\ LC & A-LC \end{bmatrix} z + \begin{bmatrix} B \\ B \end{bmatrix} u$$

La siguiente función construye este sistema para cualquier planta $(A,B,C)$ y ganancia $L$:

In [ ]:
# planta_con_observador(A, B, C, L)
# Sistema aumentado z = [x; x̂] que permite simular simultáneamente
# la planta y el observador con la misma entrada u.
function planta_con_observador(A, B, C, L)
    n  = size(A, 1)
    nu = size(B, 2)
    Aa = [A            zeros(n, n);
          L*C          A - L*C]
    Ba = [B; B]
    return ss(Aa, Ba, Matrix{Float64}(I, 2n, 2n), zeros(2n, nu))
end

In [ ]:
# Simulación: condiciones iniciales del sistema real h₁ = 0.25 m, h₂ = 0.5 m
# y el observador parte de cero (no conoce el estado inicial real)
sys_aug = planta_con_observador(A, B, C, L)

t  = 0:0.01:8
u  = ones(1, length(t))          # entrada escalón u(t) = 1
x0 = [0.15, 0.5]                  # estado inicial real
z0 = [x0; zeros(2)]               # [x; x̂]

res = lsim(sys_aug, u, t; x0 = z0)
x  = res.x[1:2, :]                # estados reales
x̂  = res.x[3:4, :];               # estados estimados

In [ ]:
p1 = plot(t, x[1, :], label = L"h_1 \;(\mathrm{real})", lw = 2)
plot!(p1, t, x̂[1, :], label = L"\hat{h}_1 \;(\mathrm{estimado})", lw = 2, ls = :dash)
ylabel!(p1, L"h_1 \; \mathrm{[m]}")

p2 = plot(t, x[2, :], label = L"h_2 \;(\mathrm{real})", lw = 2)
plot!(p2, t, x̂[2, :], label = L"\hat{h}_2 \;(\mathrm{estimado})", lw = 2, ls = :dash)
ylabel!(p2, L"h_2 \; \mathrm{[m]}")
xlabel!(p2, L"t \; \mathrm{[s]}")

plot(p1, p2, layout = (2, 1), size = (700, 500),
     plot_title = "Funcionamiento del observador (tanques)")

### Actividad 1

1. ¿Qué pasa si los polos del observador se colocan en $P_{obs}= \begin{bmatrix} \mathrm{Re}(s_1) & \mathrm{Re}(s_2)\end{bmatrix}$, siendo $s_1$ y $s_2$ los polos de lazo abierto del sistema?
2. ¿Qué pasa si los polos del observador se colocan en $P_{obs}= \begin{bmatrix} \mathrm{Re}(s_1)/10 & \mathrm{Re}(s_2)/10\end{bmatrix}$?
3. ¿Qué pasa si los polos del observador se colocan en $P_{obs}= \begin{bmatrix} 20\,\mathrm{Re}(s_1) & 20\,\mathrm{Re}(s_2)\end{bmatrix}$?

Ajuste las condiciones iniciales del sistema real en $h_1=0.25\,$m y $h_2=0.5\,$m. Diseñe, simule y concluya una vez realizados los 3 casos. Redactar un pequeño párrafo de síntesis de los experimentos realizados.

*Sugerencia:* basta con recalcular `L` con los nuevos polos y repetir la simulación anterior, por ejemplo:

```julia
polos_obs = real.(polos)            # caso 1
L = Matrix(place(A', C', polos_obs)')
```

### Actividad 2

Diseñar y simular un controlador para el sistema de tanques con polos de lazo cerrado en $s_{1,2}=-1\pm j$. En esta parte vamos a usar el observador para estimar los estados y controlar el sistema, es decir, la señal de control va a estar dada por:

$$u = k_r\,r-K\, \hat{x}$$

Diseño del controlador:

In [ ]:
polos_lc = [-1 + im, -1 - im]
K  = place(A, B, polos_lc)
kr = (-C * inv(A - B*K) * B)[1]^-1
println("K = ", K)
println("kr = ", kr)

En lugar del modelo de Simulink, el lazo cerrado con observador se puede simular con el sistema aumentado $z=[x;\ \hat{x}]$. Sustituyendo $u = k_r r - K\hat{x}$:

$$\dot{z} = \begin{bmatrix} A & -BK \\ LC & A-LC-BK \end{bmatrix} z + \begin{bmatrix} B\,k_r \\ B\,k_r \end{bmatrix} r$$

In [ ]:
# lazo_cerrado_con_observador(A, B, C, K, L, kr)
# Lazo cerrado con realimentación de los estados estimados: u = kr·r − K·x̂
# Estado aumentado z = [x; x̂]; la entrada del sistema es la referencia r
function lazo_cerrado_con_observador(A, B, C, K, L, kr)
    n  = size(A, 1)
    Aa = [A          -B*K;
          L*C         A - L*C - B*K]
    Ba = [B * kr; B * kr]
    return ss(Aa, Ba, Matrix{Float64}(I, 2n, 2n), zeros(2n, 1))
end

In [ ]:
# Simulación del lazo cerrado: referencia r = 0.5 m para h₂,
# condiciones iniciales reales h₁ = 0.25 m, h₂ = 0.5 m, observador en cero
L = Matrix(place(A', C', polos * 10)')    # observador del diseño inicial

sys_lc = lazo_cerrado_con_observador(A, B, C, K, L, kr)

t  = 0:0.01:8
r  = 0.5 * ones(1, length(t))
z0 = [0.25, 0.5, 0.0, 0.0]

res = lsim(sys_lc, r, t; x0 = z0)
x, x̂ = res.x[1:2, :], res.x[3:4, :]
u = kr .* r' .- (K * x̂)'              # reconstruimos la señal de control

p1 = plot(t, x', label = [L"h_1" L"h_2"], lw = 2)
plot!(p1, t, x̂', label = [L"\hat{h}_1" L"\hat{h}_2"], lw = 2, ls = :dash)
hline!(p1, [0.5], label = L"r", color = :gray, ls = :dot)
ylabel!(p1, "nivel [m]")

p2 = plot(t, u, label = L"u(t)", lw = 2, color = :black)
xlabel!(p2, L"t \; \mathrm{[s]}"); ylabel!(p2, L"u")

plot(p1, p2, layout = (2, 1), size = (700, 500),
     plot_title = "Control por realimentación de estados estimados")

Pruebe el controlador para los casos (2) y (3) de la **Actividad 1** y concluya (basta con recalcular `L` y repetir la simulación).

---

# Estimación de estado en un segway

Recordemos que el modelo matemático de un robot de 2 ruedas (tipo segway) linealizado en el equilibrio $x=0$, está dado por:

$$\begin{bmatrix}\dot{x}_1 \\ \dot{x}_2 \\ \dot{x}_3 \\ \dot{x}_4\end{bmatrix}=
\begin{bmatrix}0 & 1 & 0 & 0 \\
0 & -(I+m\,l^2)b/p & (m^2gl^2)/p & 0 \\
0 & 0 & 0 & 1 \\
0 & -(mlb)/p & mgl(M+m)/p & 0\end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \\ x_4\end{bmatrix}
+ \begin{bmatrix}0 \\ (I+ml^2)/p \\ 0 \\ ml/p\end{bmatrix} u$$

Las variables de estado son:

- $x_1$: posición lineal del robot
- $x_2$: velocidad lineal del robot
- $x_3$: ángulo de inclinación del robot
- $x_4$: velocidad angular en la inclinación del robot
- $u$: torque aplicado por el motor

A continuación está el código para definir este modelo como sistema lineal en variables de estado.

> **Nota (Julia):** en Julia `I` está reservado para la matriz identidad de `LinearAlgebra`, así que llamamos `Ip` al momento de inercia del péndulo.

In [ ]:
# Parámetros del modelo
M  = 1.0;     # masa del chasis
m  = 0.2;     # masa de las ruedas y el eje
b  = 0.1;     # estimación del coeficiente de fricción viscosa (N·m·s)
Ip = 0.0005;  # momento de inercia del péndulo
g  = 9.8;     # aceleración de la gravedad (m/s²)
l  = 0.125;   # distancia al centro de masa del péndulo

p = Ip*(M + m) + M*m*l^2;   # denominador de las matrices A y B

A = [0   1                  0                0;
     0  -(Ip + m*l^2)*b/p   (m^2*g*l^2)/p    0;
     0   0                  0                1;
     0  -(m*l*b)/p          m*g*l*(M + m)/p  0]

B = [0; (Ip + m*l^2)/p; 0; m*l/p;;]

C = [1.0 0 0 0]

robot = ss(A, B, C, 0)

Como vimos hace un momento, un sistema es observable si y solamente si

$$\mathcal{W}_o=\begin{bmatrix} C \\ CA \\ \vdots \\ CA^{n-1}\end{bmatrix}$$

tiene rango $n$. A continuación vamos a ver qué pasa con la observabilidad en el caso de poner **un solo sensor** para estimar los 4 estados del sistema. Vamos a elegir alternativamente un sensor en cada caso.

## Caso 1:

In [ ]:
C1 = [1.0 0 0 0]    # sensor de posición lineal x₁
Wo = obsv(A, C1)
display(Wo)
println("rank(Wo) = ", rank(Wo))

Para explorar los demás casos, basta con cambiar la matriz de medición (descomente el sensor que quiera probar):

In [ ]:
# Seleccione la medición que quiere probar:
Cs = [1.0 0 0 0]    # x₁: posición lineal
# Cs = [0.0 1 0 0]  # x₂: velocidad lineal
# Cs = [0.0 0 1 0]  # x₃: ángulo de inclinación
# Cs = [0.0 0 0 1]  # x₄: velocidad angular de inclinación

Wo = obsv(A, Cs)
display(round.(Wo, digits = 4))
println("rank(Wo) = ", rank(Wo))

### Actividad 3

- ¿Con cuáles sensores individuales podemos estimar todo el estado del sistema?


### Actividad 4

Diseñe un controlador LQR para el robot (suponga, por ejemplo, $u_{max} = 10$) y un observador que ubique los polos 10 veces más rápidos que los del controlador LQR. Considere dos casos:

**a)** Medimos $x_1$ (posición lineal) y $x_4$ (velocidad angular de inclinación) para estimar los demás estados del segway.
**b)** Medimos $x_1$ (posición lineal) 



In [ ]:
C = [1.0 0 0 0;     # ejemplo de matriz C cuando escojo las mediciones de x₁ y x₄
     0.0 0 0 1]

# Controlador LQR
Q = diagm([1.0, 1.0, 1.0, 1.0])
R = fill(0.01, 1, 1)
K = lqr(Continuous, A, B, Q, R)
println("K = ", K)

# Polos de lazo cerrado y observador 5 veces más rápido
polos_lc = eigvals(A - B*K)
println("polos de lazo cerrado: ", polos_lc)

L = Matrix(place(A', C', polos_lc * 5)')

# Ganancia de pre-compensación para seguir referencia en x₁
C1 = [1.0 0 0 0]
kr = (-C1 * inv(A - B*K) * B)[1]^-1
println("kr = ", kr)



Simulamos el lazo cerrado completo (planta + observador + ley de control $u = k_r r - K\hat{x}$) reutilizando la función `lazo_cerrado_con_observador`. El robot parte con una inclinación inicial $x_3 = 0.1$ rad y el observador parte de cero:

In [ ]:
sys_segway = lazo_cerrado_con_observador(A, B, C, K, L, kr)

t  = 0:0.001:5
r  = zeros(1, length(t))                   
z0 = [0.0, 0.0, 0.2, 0.0,   0.0, 0.0, 0.0, 0.0]   # [x; x̂]

res = lsim(sys_segway, r, t; x0 = z0)
x, x̂ = res.x[1:4, :], res.x[5:8, :]
u = kr .* r' .- (K * x̂)'

etiquetas  = [L"x_1\,\mathrm{[m]}" L"x_2\,\mathrm{[m/s]}" L"x_3\,\mathrm{[rad]}" L"x_4\,\mathrm{[rad/s]}"]
ps = [plot(t, x[i, :], label = "real", lw = 2, ylabel = etiquetas[i]) for i in 1:4]
for i in 1:4
    plot!(ps[i], t, x̂[i, :], label = "estimado", lw = 2, ls = :dash)
end
xlabel!(ps[3], L"t\,\mathrm{[s]}"); xlabel!(ps[4], L"t\,\mathrm{[s]}")

plot(ps..., layout = (2, 2), size = (900, 550),
     plot_title = "Segway: LQR + observador (sensores x₁ y x₄)")

In [ ]:
plot(t, u, label = L"u(t)", lw = 2, color = :black,
     xlabel = L"t\,\mathrm{[s]}", ylabel = L"u\,\mathrm{[N\cdot m]}",
     title = "Señal de control (verifique |u| ≤ 10)", size = (700, 300))

Pruebe este modelo (que ya incluye el observador) y evalúe el funcionamiento para los dos tipos de observación propuestos. Por ejemplo, para el caso **b)** con otros sensores, basta con redefinir la matriz `C`, recalcular `L` y repetir la simulación.